# 05 – Finjustere mDeBERTa (kjøres i Google Colab)

Baselinen (TF-IDF) og ProtectAI har hver sine svakheter. Her trener vi vår egen transformer, [`microsoft/mdeberta-v3-base`](https://huggingface.co/microsoft/mdeberta-v3-base). Den er flerspråklig (forstår norsk) og har samme arkitektur som ProtectAI.

Den trenes på nøyaktig samme treningsmiks som baselinen, inkludert de maskinoversatte norske dataene. Den evalueres med `evaluate_by_source`, slik at tallene kan sammenlignes direkte med notebook 02–04.

**Slik kjører du den:**
1. Åpne notebooken i Colab: *File → Upload notebook*, eller åpne den fra GitHub.
2. *Runtime → Change runtime type → T4 GPU*.
3. *Runtime → Run all*. Alt tar ca. 45–60 minutter: data ~5 min, oversettelse ~5 min, trening ~20–25 min, evaluering ~5 min.
4. Last ned `mdeberta_scores.parquet` til slutt (siste celle), og legg den i `models/` lokalt.

## Oppsett

In [ ]:
import torch
assert torch.cuda.is_available(), "Ingen GPU. Velg Runtime → Change runtime type → T4 GPU."
print(torch.cuda.get_device_name(0))

In [ ]:
!git clone -q https://github.com/muhayudinawan/prompt-injection-detector.git
%cd prompt-injection-detector
!pip install -q datasets transformers accelerate sentencepiece scikit-learn pyarrow

In [ ]:
import os
import sys
import warnings
from pathlib import Path

os.environ["WANDB_DISABLED"] = "true"
warnings.filterwarnings("ignore")

import datasets
import numpy as np
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
datasets.disable_progress_bars()

from data import load_all, load_calibration_no, load_own_testsets, load_training_data, TEST_ONLY_SOURCES
from evaluate import evaluate_by_source, threshold_for_fpr

## Data

Samme steg som notebook 01 og `src/translate.py`. I Colab finnes ikke `data/` fra før, så vi bygger den på nytt.

In [ ]:
(ROOT / "data").mkdir(exist_ok=True)
_, combined = load_all()
combined.to_parquet(ROOT / "data" / "combined.parquet", index=False)
print(f"{len(combined):,} rader i combined.parquet")

In [ ]:
!python src/translate.py

In [ ]:
from sklearn.model_selection import train_test_split

df = load_training_data(ROOT)
train_all = df[df["split"] == "train"]
train, val = train_test_split(
    train_all, test_size=0.1, random_state=0,
    stratify=train_all["source"] + train_all["label"].astype(str),
)
test = df[df["split"] == "test"]
own = load_own_testsets(ROOT)
cal = load_calibration_no(ROOT)
print(f"train: {len(train):,}   val: {len(val):,}   test: {len(test):,}   egne testsett: {len(own):,}   kalibrering: {len(cal):,}")
print(f"andel angrep i train: {train['label'].mean():.1%}")

## Trening

To epoker med læringsrate 2e-5 er et vanlig utgangspunkt for DeBERTa. Tekstene kuttes ved 256 tokens for å holde treningen rask. Det dekker de aller fleste meldingene.

`FP16` er slått av med vilje. Med FP16 eksploderte tallene i mDeBERTa under treningen (`loss = nan`), og alle scorer ble `NaN`. Av samme grunn gjøres modellen om til float32 med `.float()`: transformers 5 laster sjekkpunktet i float16, selv når FP16 er av. Uten FP16 tar treningen ca. 10–25 minutter på en T4, men den er stabil.

In [ ]:
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

MODEL = "microsoft/mdeberta-v3-base"
MAX_LEN = 256
FP16 = False

tokenizer = AutoTokenizer.from_pretrained(MODEL)

def to_dataset(d):
    ds = datasets.Dataset.from_dict({"text": d["text"].tolist(), "label": d["label"].astype(int).tolist()})
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=["text"])

train_ds, val_ds = to_dataset(train), to_dataset(val)
# transformers 5 laster sjekkpunktet i float16. Det må være float32, ellers blir loss = nan.
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).float()

args = TrainingArguments(
    output_dir="mdeberta-out",
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=int(0.06 * 2 * len(train_ds) / 16),  # 6 % av stegene; transformers 5 har fjernet warmup_ratio
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=FP16,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    logging_nan_inf_filter=False,  # vis nan i loggen i stedet for å skjule den
    report_to="none",
    seed=0,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tokenizer))
trainer.train()

## Score alle tekstene

In [ ]:
import torch

@torch.inference_mode()
def score(texts, batch_size=64):
    model.eval()
    texts = list(texts)
    order = np.argsort([len(t) for t in texts])
    out = np.empty(len(texts))
    for i in range(0, len(texts), batch_size):
        idx = order[i:i + batch_size]
        enc = tokenizer([texts[j] for j in idx], truncation=True, max_length=MAX_LEN,
                        padding=True, return_tensors="pt").to(model.device)
        out[idx] = model(**enc).logits.float().softmax(-1)[:, 1].cpu().numpy()
    return out

val_scores = score(val["text"])
test_scores = score(test["text"])
own_scores = score(own["text"])
cal_scores = score(cal["text"])
if np.isnan(np.concatenate([val_scores, test_scores, own_scores, cal_scores])).any():
    raise ValueError("Scorene er NaN: treningen feilet (loss = nan). Sjekk at FP16 = False og kjør treningen på nytt.")
threshold = threshold_for_fpr(cal["label"], cal_scores, max_fpr=0.02)
print(f"Terskel for 2 % falske positive på kalibreringssettet: {threshold:.4f}")

## Baseline på samme data

For en rettferdig sammenligning trener vi baselinen fra notebook 02 på nøyaktig de samme dataene her. Det tar et par minutter.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, make_pipeline

baseline = make_pipeline(
    FeatureUnion([
        ("ord", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000, sublinear_tf=True)),
        ("tegn", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, max_features=300_000, sublinear_tf=True)),
    ]),
    LogisticRegression(max_iter=2000, C=4.0, class_weight="balanced"),
).fit(train["text"], train["label"])
base_cal = baseline.predict_proba(cal["text"])[:, 1]
base_threshold = threshold_for_fpr(cal["label"], base_cal, max_fpr=0.02)
base_test = baseline.predict_proba(test["text"])[:, 1]
base_own = baseline.predict_proba(own["text"])[:, 1]

## Resultater

ROC AUC er hovedmålet, fordi den ikke avhenger av terskelen.

In [ ]:
def compare(d, a, ta, b, tb, cols=("recall", "fpr", "roc_auc")):
    return pd.concat({
        "mDeBERTa": evaluate_by_source(d, a, ta)[list(cols)],
        "baseline": evaluate_by_source(d, b, tb)[list(cols)],
    }, axis=1)

print("Egne testsett, per språk:")
display(compare(own.assign(source=own["language"]), own_scores, threshold, base_own, base_threshold,
                cols=("precision", "recall", "fpr", "roc_auc")))
print("Offentlige testsett:")
compare(test, test_scores, threshold, base_test, base_threshold)

In [ ]:
flag = own.assign(
    mdeberta=(own_scores > threshold).astype(int),
    baseline=(base_own > base_threshold).astype(int),
)
flag.groupby(["label", "category", "language"])[["mdeberta", "baseline"]].mean().unstack("language").round(3)

## Feilanalyse på norsk

In [ ]:
pd.set_option("display.max_colwidth", 200)
no = own.assign(score=own_scores, pred=(own_scores > threshold).astype(int))
no = no[no["language"] == "no"]
print("Angrep som slapp gjennom:")
display(no[(no["label"] == 1) & (no["pred"] == 0)].nsmallest(10, "score")[["id", "score", "text"]])
print("Vanlige meldinger som ble blokkert:")
no[(no["label"] == 0) & (no["pred"] == 1)].nlargest(10, "score")[["id", "score", "text"]]

## Lagre og laste ned

Scorene lagres, slik at de kan sammenlignes lokalt uten å trene på nytt. Legg `mdeberta_scores.parquet` i `models/` i prosjektet ditt.

Vil du beholde selve modellen (ca. 1 GB), kan du koble til Google Drive og lagre den der. Se den siste, valgfrie cellen.

In [ ]:
import hashlib

def key(t):
    return hashlib.sha1(t.encode("utf-8")).hexdigest()

scores = pd.DataFrame({
    "key": [key(t) for t in pd.concat([val["text"], test["text"], own["text"], cal["text"]])],
    "score": np.concatenate([val_scores, test_scores, own_scores, cal_scores]),
}).drop_duplicates("key")
scores.attrs["threshold"] = threshold
scores.to_parquet("mdeberta_scores.parquet", index=False)

from google.colab import files
files.download("mdeberta_scores.parquet")

In [ ]:
# Valgfritt: lagre modellen i Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# trainer.save_model("/content/drive/MyDrive/prompt-injection-detector/mdeberta")
# tokenizer.save_pretrained("/content/drive/MyDrive/prompt-injection-detector/mdeberta")